# 01 - Extract CBAM Default Values (xlsx)

## Purpose
Extract CBAM definitive period default values from the EU Commission xlsx file.
One tab per country, flatten into a single standardized table.

## Input
`data/raw/DVs as adopted_v20260204.xlsx`

## Output
`data/processed/cbam_defaults.csv`

In [5]:
import pandas as pd
from pathlib import Path

# Use the file's actual location
xlsx_path = Path("/Users/milcahmaryjoseph/Documents/GitHub/cbam-analysis/data/raw/DVs as adopted_v20260204 .xlsx")

print(f"File exists: {xlsx_path.exists()}")

xl = pd.ExcelFile(xlsx_path)
sheet_names = xl.sheet_names

print(f"Total sheets: {len(sheet_names)}")
print(f"\nFirst 10 sheets: {sheet_names[:10]}")
print(f"\nLast 10 sheets: {sheet_names[-10:]}")

File exists: True
Total sheets: 122

First 10 sheets: ['Overview', 'Version History', 'Albania', 'Algeria', 'Angola', 'Argentina', 'Armenia', 'Australia', 'Azerbaijan', 'Bangladesh']

Last 10 sheets: ['United Kingdom', 'United States', 'Uruguay', 'Uzbekistan', 'Venezuela', 'Vietnam', 'Yemen', 'Zambia', 'Zimbabwe', '_Other Countries and Territorie']


In [6]:
# Inspect a single country tab to understand structure
sample = pd.read_excel(xlsx_path, sheet_name="India", header=None)
print(sample.shape)
print(sample.head(20).to_string())

(296, 9)
                  0                                                                                                                                                                                                                                                                                1                                  2                                    3                                 4                                         5                                         6                                                     7                                                8
0             India                                                                                                                                                                                                                                                                              NaN                                NaN                                  NaN                               NaN       

## Structure Notes
- Row 0: country name (skip)
- Row 1: column headers
- Data rows interspersed with section header rows (material category names)
  where cols 2-8 are NaN, used for grouping only
- Mark-up % rows also interspersed, not data
- Some CN codes have "see below" values, deferred to sub-rows
- Row count varies by country
- Material category can be inferred from CN code, but is present as section headers
- Column 8 (Underlying production route) is sparsely populated

In [7]:
# Read India tab with row 1 as header
df_sample = pd.read_excel(xlsx_path, sheet_name="India", header=1)

print("Shape:", df_sample.shape)
print("\nColumns:", df_sample.columns.tolist())
print("\nFirst 20 rows:")
df_sample.head(20)

Shape: (294, 9)

Columns: ['Product CN Code', 'Description', 'Default Value\n(direct emissions)', 'Default Value\n(indirect emissions)', 'Default Value\n(total emissions)', '2026\nDefault Value\n(including mark-up)', '2027\nDefault Value\n(including mark-up)', '2028 and onwards\nDefault Value\n(including mark-up)', 'Underlying production route determining CBAM BM']

First 20 rows:


,Product CN Code,Description,Default Value\n(direct emissions),Default Value\n(indirect emissions),Default Value\n(total emissions),2026\nDefault Value\n(including mark-up),2027\nDefault Value\n(including mark-up),2028 and onwards\nDefault Value\n(including mark-up),Underlying production route determining CBAM BM
0,Cement,NaN,NaN,NaN,NaN,10% mark-up,20% mark-up,30% mark-up,
1,2507 00 80,Calcined clay,–,–,–,_,_,_,NaN
2,2523 10 00,White clinker,1.35,0.07,1.41,1.551,1.692,1.833,(B)
3,2523 10 00,Grey clinker,1.39,0.05,1.44,1.584,1.728,1.872,(A)
4,2523 21 00,White Portland cement,1.33,0.14,1.47,1.617,1.764,1.911,
5,2523 29 00,Grey Portland cement,1.39,0.09,1.48,1.628,1.776,1.924,
6,2523 90 00,White hydraulic cements,–,–,–,_,_,_,NaN
7,2523 90 00,Grey hydraulic cements,–,–,–,_,_,_,NaN
8,2523 30 00,Aluminous cement,2.05,0.24,2.29,2.519,2.748,2.977,
9,Fertilisers,NaN,NaN,NaN,NaN,1% mark-up,1% mark-up,1% mark-up,


In [8]:
# Define the non-data row patterns to filter out
# Test our filtering logic on the India sample

def is_data_row(row):
    """Returns True if row contains actual emission data."""
    cn = str(row["Product CN Code"]).strip()
    total = row["Default Value\n(total emissions)"]
    
    # Skip section header rows (CN code is a word like "Cement")
    if not any(char.isdigit() for char in cn):
        return False
    
    # Skip rows where total emissions is "see below", "–", or NaN
    if pd.isna(total):
        return False
    if str(total).strip() in ["see below", "–", "_", "-"]:
        return False
    
    # Skip rows where total is not numeric
    try:
        float(total)
    except (ValueError, TypeError):
        return False
    
    return True

# Apply to India sample
mask = df_sample.apply(is_data_row, axis=1)
df_india_clean = df_sample[mask].copy()

print(f"Original rows: {len(df_sample)}")
print(f"After filtering: {len(df_india_clean)}")
print(f"\nFirst 10 clean rows:")
df_india_clean.head(10)

Original rows: 294
After filtering: 256

First 10 clean rows:


,Product CN Code,Description,Default Value\n(direct emissions),Default Value\n(indirect emissions),Default Value\n(total emissions),2026\nDefault Value\n(including mark-up),2027\nDefault Value\n(including mark-up),2028 and onwards\nDefault Value\n(including mark-up),Underlying production route determining CBAM BM
2,2523 10 00,White clinker,1.35,0.07,1.41,1.551,1.692,1.833,(B)
3,2523 10 00,Grey clinker,1.39,0.05,1.44,1.584,1.728,1.872,(A)
4,2523 21 00,White Portland cement,1.33,0.14,1.47,1.617,1.764,1.911,
5,2523 29 00,Grey Portland cement,1.39,0.09,1.48,1.628,1.776,1.924,
8,2523 30 00,Aluminous cement,2.05,0.24,2.29,2.519,2.748,2.977,
10,2808 00 00,Nitric acid; sulphonitric acids,1.94,0.07,2.01,2.0301,2.0301,2.0301,
11,28141000,Anhydrous ammonia,3.06,0.22,3.28,3.3128,3.3128,3.3128,
12,28142000,Ammonia in aqueous solution,0.92,0.07,0.99,0.9999,0.9999,0.9999,
13,2834 21 00,Nitrate of potassium,1.86,0.1,1.96,1.9796,1.9796,1.9796,
15,3102 10 12,"Urea in aqueous solution, containing >45% nitr...",0.68,0.06,0.74,0.7474,0.7474,0.7474,


In [9]:
# Rename columns to clean snake_case names
column_mapping = {
    "Product CN Code": "cn_code",
    "Description": "description",
    "Default Value\n(direct emissions)": "direct_emissions",
    "Default Value\n(indirect emissions)": "indirect_emissions",
    "Default Value\n(total emissions)": "total_emissions",
    "2026\nDefault Value\n(including mark-up)": "default_2026",
    "2027\nDefault Value\n(including mark-up)": "default_2027",
    "2028 and onwards\nDefault Value\n(including mark-up)": "default_2028_onwards",
    "Underlying production route determining CBAM BM": "production_route"
}

df_india_clean = df_india_clean.rename(columns=column_mapping)

# Add country column
df_india_clean["country"] = "India"

# Convert value columns to numeric, coercing any remaining non-numeric to NaN
value_cols = ["direct_emissions", "indirect_emissions", "total_emissions",
              "default_2026", "default_2027", "default_2028_onwards"]

for col in value_cols:
    df_india_clean[col] = pd.to_numeric(df_india_clean[col], errors="coerce")

print(df_india_clean.dtypes)
print(f"\nShape: {df_india_clean.shape}")
df_india_clean.head(5)

cn_code                  object
description                 str
direct_emissions        float64
indirect_emissions      float64
total_emissions         float64
default_2026            float64
default_2027            float64
default_2028_onwards    float64
production_route            str
country                     str
dtype: object

Shape: (256, 10)


,cn_code,description,direct_emissions,indirect_emissions,total_emissions,default_2026,default_2027,default_2028_onwards,production_route,country
2,2523 10 00,White clinker,1.35,0.07,1.41,1.551,1.692,1.833,(B),India
3,2523 10 00,Grey clinker,1.39,0.05,1.44,1.584,1.728,1.872,(A),India
4,2523 21 00,White Portland cement,1.33,0.14,1.47,1.617,1.764,1.911,,India
5,2523 29 00,Grey Portland cement,1.39,0.09,1.48,1.628,1.776,1.924,,India
8,2523 30 00,Aluminous cement,2.05,0.24,2.29,2.519,2.748,2.977,,India


In [11]:
country_sheets = [s for s in sheet_names if s not in ["Overview", "Version History", "_Other Countries and Territorie"]]

print(f"Countries to extract: {len(country_sheets)}")

all_countries = []

for country in country_sheets:
    try:
        df = pd.read_excel(xlsx_path, sheet_name=country, header=1)
        
        # Rename BEFORE filtering
        df = df.rename(columns=column_mapping)
        
        # Now filter using renamed columns
        def is_data_row_renamed(row):
            cn = str(row["cn_code"]).strip()
            total = row["total_emissions"]
            if not any(char.isdigit() for char in cn):
                return False
            if pd.isna(total):
                return False
            if str(total).strip() in ["see below", "–", "_", "-"]:
                return False
            try:
                float(total)
            except (ValueError, TypeError):
                return False
            return True

        mask = df.apply(is_data_row_renamed, axis=1)
        df_clean = df[mask].copy()
        df_clean["country"] = country
        for col in value_cols:
            df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")
        all_countries.append(df_clean)
    except Exception as e:
        print(f"Error on {country}: {e}")

df_all = pd.concat(all_countries, ignore_index=True)

print(f"\nTotal rows extracted: {len(df_all)}")
print(f"Countries successfully extracted: {len(all_countries)}")
print(f"Countries with errors: {len(country_sheets) - len(all_countries)}")

Countries to extract: 119

Total rows extracted: 10671
Countries successfully extracted: 119
Countries with errors: 0


In [12]:
# Sanity checks before saving
print("=== Column order ===")
print(df_all.columns.tolist())

print("\n=== Sample countries present ===")
print(df_all["country"].unique()[:10])

print("\n=== Row counts per country (sample) ===")
print(df_all.groupby("country").size().sort_values(ascending=False).head(10))

print("\n=== Value ranges for total_emissions ===")
print(df_all["total_emissions"].describe())

print("\n=== Any nulls? ===")
print(df_all.isnull().sum())

=== Column order ===
['cn_code', 'description', 'direct_emissions', 'indirect_emissions', 'total_emissions', 'default_2026', 'default_2027', 'default_2028_onwards', 'production_route', 'country', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12']

=== Sample countries present ===
<StringArray>
[   'Albania',    'Algeria',     'Angola',  'Argentina',    'Armenia',
  'Australia', 'Azerbaijan', 'Bangladesh',    'Bahrain',    'Belarus']
Length: 10, dtype: str

=== Row counts per country (sample) ===
country
China             259
United States     258
India             256
South Korea       255
Japan             254
Russia            254
Canada            254
Kazakhstan        253
Algeria           253
United Kingdom    252
dtype: int64

=== Value ranges for total_emissions ===
count    10671.000000
mean         2.584043
std          1.800383
min          0.000000
25%          1.440000
50%          2.330000
75%          3.210000
max         26.640000
Name: total_emissions, dtype: fl

In [13]:
# Inspect unnamed columns
print("=== Unnamed: 9 non-null values ===")
print(df_all["Unnamed: 9"].dropna().unique()[:20])

print("\n=== Unnamed: 10 non-null values ===")
print(df_all["Unnamed: 10"].dropna().unique()[:20])

print("\n=== Unnamed: 11 non-null values ===")
print(df_all["Unnamed: 11"].dropna().unique()[:20])

print("\n=== Unnamed: 12 non-null values ===")
print(df_all["Unnamed: 12"].dropna().unique()[:20])

=== Unnamed: 9 non-null values ===
[]

=== Unnamed: 10 non-null values ===
[]

=== Unnamed: 11 non-null values ===
[7.612000e+03 7.614000e+03 7.610900e+05 7.604101e+07 7.604109e+07
 7.604210e+07 7.604291e+07 7.604299e+07 7.609000e+07 7.610100e+07
 7.610901e+07 7.610909e+07 7.611000e+07 7.613000e+07 7.616100e+07]

=== Unnamed: 12 non-null values ===
[4.53151485 3.70257921 3.721      3.059     ]


In [14]:
# Check which rows have values in Unnamed: 11 and 12
spill_rows = df_all[df_all["Unnamed: 11"].notna()]
print(f"Rows with Unnamed: 11 values: {len(spill_rows)}")
print("\nSample of these rows:")
print(spill_rows[["country", "cn_code", "description", "total_emissions", 
                   "Unnamed: 11", "Unnamed: 12"]].head(10).to_string())

Rows with Unnamed: 11 values: 15

Sample of these rows:
                     country   cn_code                                                                                                                                                                  description  total_emissions  Unnamed: 11  Unnamed: 12
1248  Bosnia and Herzegovina  76042910                                                                                                                                                                Bars and rods         3.059000       7612.0     4.531515
1249  Bosnia and Herzegovina  76042990                                                                                                                                                                     Profiles         3.059000       7614.0     3.702579
1250  Bosnia and Herzegovina      7605                                                                                                                                             

In [15]:
# Check which rows have values in Unnamed: 11 and 12
spill_rows = df_all[df_all["Unnamed: 11"].notna()]
print(f"Rows with Unnamed: 11 values: {len(spill_rows)}")
print("\nSample of these rows:")
print(spill_rows[["country", "cn_code", "description", "total_emissions", 
                   "Unnamed: 11", "Unnamed: 12"]].tail(10).to_string())

Rows with Unnamed: 11 values: 15

Sample of these rows:
                     country   cn_code                                                                                                                                                                                                                                                                                                     description  total_emissions  Unnamed: 11  Unnamed: 12
1253  Bosnia and Herzegovina      7608                                                                                                                                                                                                                                                                                       Aluminium tubes and pipes         3.721000   76042100.0     3.059000
1254  Bosnia and Herzegovina  76090000                                                                                                                                      

## Bosnia and Herzegovina: Spill Column Investigation

Rows 48-62 in the Bosnia and Herzegovina tab have data in columns L and M
(Unnamed: 11 and Unnamed: 12 in pandas). Investigation confirms these are
duplicates of data already present correctly in the main table columns A and C.

Pattern confirmed:
- Unnamed: 11 contains CN codes that exist in cn_code column
- Unnamed: 12 contains emission values that match total_emissions for those CN codes
- Likely caused by a copy-paste formatting error in the source xlsx

Decision: Drop Unnamed: 9, 10, 11, 12. Main table data for Bosnia and
Herzegovina is correct and complete. No data loss from dropping these columns.

![Bosnia spill columns 1](bosnia_spill_1.png)
![Bosnia spill columns 2](bosnia_spill_2.png)
![Bosnia spill columns 3](bosnia_spill_3.png)

In [20]:
# Confirm spill values are duplicates of existing main table data

# Display as table with final count
results = []

for _, row in spill_rows.iterrows():
    spill_cn = str(int(row["Unnamed: 11"]))
    spill_val = round(float(row["Unnamed: 12"]), 3)
    
    match = bosnia[bosnia["cn_clean"] == spill_cn]
    
    if len(match) > 0:
        main_val = round(float(match["total_emissions"].values[0]), 3)
        is_duplicate = abs(spill_val - main_val) < 0.01
        results.append({
            "spill_cn": spill_cn,
            "spill_value": spill_val,
            "main_table_cn": match["cn_code"].values[0],
            "main_table_value": main_val,
            "is_duplicate": is_duplicate
        })
    else:
        results.append({
            "spill_cn": spill_cn,
            "spill_value": spill_val,
            "main_table_cn": "NO MATCH",
            "main_table_value": None,
            "is_duplicate": False
        })

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))
print(f"\nTotal spill rows: {len(df_results)}")
print(f"Confirmed duplicates: {df_results['is_duplicate'].sum()}")
print(f"Not duplicates: {(~df_results['is_duplicate']).sum()}")

spill_cn  spill_value  main_table_cn  main_table_value  is_duplicate
    7612        4.532           7612             4.532          True
    7614        3.703           7614             3.703          True
  761090        3.721         761090             3.721          True
76041010        3.059       76041010             3.059          True
76041090        3.059       76041090             3.059          True
76042100        3.059       76042100             3.059          True
76042910        3.059       76042910             3.059          True
76042990        3.059       76042990             3.059          True
76090000        3.721       76090000             3.721          True
76101000        3.721       76101000             3.721          True
76109010        3.721       76109010             3.721          True
76109090        3.721       76109090             3.721          True
76110000        4.532       76110000             4.532          True
76130000        4.532       761300

In [21]:
# Drop unnamed columns, confirmed as duplicates of main table data
cols_to_keep = [c for c in df_all.columns if not c.startswith("Unnamed")]
df_all = df_all[cols_to_keep]

# Reorder so country is first
df_all = df_all[["country", "cn_code", "description", "direct_emissions",
                  "indirect_emissions", "total_emissions", "default_2026",
                  "default_2027", "default_2028_onwards", "production_route"]]

# Save to processed
output_path = Path("/Users/milcahmaryJoseph/Documents/GitHub/cbam-analysis/data/processed/cbam_defaults.csv")
df_all.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")
print(f"Final shape: {df_all.shape}")
print(f"\nSample:")
df_all.head(5)

Saved to: /Users/milcahmaryJoseph/Documents/GitHub/cbam-analysis/data/processed/cbam_defaults.csv
Final shape: (10671, 10)

Sample:


,country,cn_code,description,direct_emissions,indirect_emissions,total_emissions,default_2026,default_2027,default_2028_onwards,production_route
0,Albania,2523 10 00,Grey clinker,0.87,0.00,0.87,0.9570,1.0440,1.1310,(A)
1,Albania,2523 29 00,Grey Portland cement,0.90,0.03,0.93,1.0230,1.1160,1.2090,NaN
2,Albania,2523 90 00,Grey hydraulic cements,0.86,0.03,0.89,0.9790,1.0680,1.1570,(A)
3,Albania,2808 00 00,Nitric acid; sulphonitric acids,2.73,0.04,2.76,2.7876,2.7876,2.7876,NaN
4,Albania,28142000,Ammonia in aqueous solution,0.65,0.03,0.68,0.6868,0.6868,0.6868,NaN
